# 🛒 Caso Práctico 1 — RetailNova

Pipeline completo: **CSV → DataFrame → Dataset → DataFrame enriquecido → Parquet particionado → Spark SQL**.

## 🏢 Contexto empresarial

**RetailNova** es una empresa de comercio electrónico que vende productos tecnológicos en España. Cada año exporta sus ventas en un fichero CSV independiente (`ventas_2022.csv`, `ventas_2023.csv`, `ventas_2024.csv`).

El equipo de datos quiere construir una **capa analítica** optimizada para que los analistas puedan consultar los resultados con **Spark SQL**.

## 🎯 Objetivos

1. Leer múltiples CSV anuales en un único DataFrame.
2. Normalizar tipos y limpiar texto.
3. Convertir a `Dataset[VentaRaw]` para aplicar lógica de negocio tipada en Scala.
4. Calcular importes (bruto, descuento, neto) y clasificar ventas con una `case class VentaEnriquecida`.
5. Volver a DataFrame y aplicar una **UDF** de riesgo comercial.
6. Añadir columnas derivadas con funciones nativas de Spark SQL.
7. Reducir a una **capa analítica** y guardarla como **Parquet particionado por `anio`/`mes`**.
8. Exponer el resultado como vista temporal y resolver consultas de negocio con **Spark SQL**.
9. **Benchmark** comparativo Avro / ORC / Parquet.

---

## ⚙️ Parte 1 — Inicialización del entorno

> Ejecuta esta celda **antes que cualquier otra** y espera al mensaje ✅.

In [1]:
import $ivy.`org.apache.spark::spark-core:4.1.1`
import $ivy.`org.apache.spark::spark-sql:4.1.1`
import $ivy.`org.apache.spark::spark-avro:4.1.1`

import org.apache.log4j.{Level, Logger}
Logger.getLogger("org").setLevel(Level.ERROR)
Logger.getLogger("akka").setLevel(Level.ERROR)

import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.functions._
import org.apache.spark.sql.types._
import org.apache.spark.sql.{Dataset, DataFrame}

import java.nio.file.{Files, Paths}
import java.nio.charset.StandardCharsets
import java.io.File

val spark = SparkSession.builder()
  .appName("RetailNova_Dataset_DataFrame_Parquet_SQL")
  .master("local[*]")
  .config("spark.sql.shuffle.partitions", "4")
  .config("spark.ui.showConsoleProgress", "false")
  .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

import spark.implicits._

println(s"✅ Spark ${spark.version} listo")
println(s"✅ Scala ${scala.util.Properties.versionNumberString} listo")

Downloaded https://repo1.maven.org/maven2/org/apache/spark/spark-avro_2.13/4.1.1/spark-avro_2.13-4.1.1.pom
Downloaded https://repo1.maven.org/maven2/javax/servlet/javax.servlet-api/3.1.0/javax.servlet-api-3.1.0.pom
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/05 13:34:26 INFO SparkContext: Running Spark version 4.1.1
26/05/05 13:34:26 INFO SparkContext: OS info Windows 11, 10.0, amd64
26/05/05 13:34:26 INFO SparkContext: Java version 17.0.18+8
26/05/05 13:34:26 INFO ResourceUtils: ==============================================================
26/05/05 13:34:26 INFO ResourceUtils: No custom resources configured for spark.driver.
26/05/05 13:34:26 INFO ResourceUtils: ==============================================================
26/05/05 13:34:26 INFO SparkContext: Submitted application: RetailNova_Dataset_DataFrame_Parquet_SQL
26/05/05 13:34:26 INFO SecurityManager: Changing view acls to: gre
26/05/05 13:34:26 INFO SecurityManager: Changing modi

✅ Spark 4.1.1 listo
✅ Scala 2.13.17 listo


import $ivy.$
import $ivy.$
import $ivy.$
import org.apache.log4j.{Level, Logger}
import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.functions._
import org.apache.spark.sql.types._
import org.apache.spark.sql.{Dataset, DataFrame}
import java.nio.file.{Files, Paths}
import java.nio.charset.StandardCharsets
import java.io.File
spark: SparkSession = org.apache.spark.sql.classic.SparkSession@3b881856
import spark.implicits._

---

## 📁 Parte 2 — Crear los CSV sintéticos

Los ficheros se crean dentro de la carpeta `data/` **relativa al notebook**, para mantener el caso autocontenido en este workspace.

In [2]:
// 2.1 Carpeta de trabajo (relativa al notebook)
val rutaBase = "data"
Files.createDirectories(Paths.get(rutaBase))

println(s"Carpeta creada o existente: $rutaBase")

Carpeta creada o existente: data


rutaBase: String = "data"
res2_1: java.nio.file.Path = data

In [3]:
// 2.2 ventas_2022.csv
val ventas2022 =
"""id_venta,fecha,id_cliente,cliente,pais,canal,categoria,producto,unidades,precio_unitario,descuento_pct
V2022-001,2022-01-15,C001,Ana García,España,web,Informática,Portátil Pro,1,950.00,5
V2022-002,2022-02-03,C002,Luis Martín,España,tienda,Periféricos,Teclado Mecánico,2,75.00,0
V2022-003,2022-03-18,C003,Marta López,España,web,Periféricos,Ratón Inalámbrico,3,29.90,10
V2022-004,2022-04-22,C004,Carlos Ruiz,Portugal,marketplace,Audio,Auriculares USB,2,59.99,5
V2022-005,2022-05-11,C005,Elena Vega,España,web,Monitores,Monitor 27,1,220.00,15
V2022-006,2022-06-30,C006,Jorge Díaz,Francia,tienda,Informática,Tablet 10,1,310.00,0
V2022-007,2022-08-09,C007,Laura Prieto,España,web,Almacenamiento,SSD 1TB,2,115.00,5
V2022-008,2022-09-25,C008,Pedro Santos,España,marketplace,Audio,Webcam HD,1,79.00,0
V2022-009,2022-10-14,C009,Sofía Ramos,Portugal,web,Informática,Portátil Air,1,780.00,8
V2022-010,2022-12-02,C010,Andrés Mora,España,tienda,Periféricos,Hub USB-C,4,35.00,0
"""

Files.write(Paths.get(s"$rutaBase/ventas_2022.csv"), ventas2022.getBytes(StandardCharsets.UTF_8))
println("✅ ventas_2022.csv creado")

✅ ventas_2022.csv creado


ventas2022: String = """id_venta,fecha,id_cliente,cliente,pais,canal,categoria,producto,unidades,precio_unitario,descuento_pct
V2022-001,2022-01-15,C001,Ana García,España,web,Informática,Portátil Pro,1,950.00,5
V2022-002,2022-02-03,C002,Luis Martín,España,tienda,Periféricos,Teclado Mecánico,2,75.00,0
V2022-003,2022-03-18,C003,Marta López,España,web,Periféricos,Ratón Inalámbrico,3,29.90,10
V2022-004,2022-04-22,C004,Carlos Ruiz,Portugal,marketplace,Audio,Auriculares USB,2,59.99,5
V2022-005,2022-05-11,C005,Elena Vega,España,web,Monitores,Monitor 27,1,220.00,15
V2022-006,2022-06-30,C006,Jorge Díaz,Francia,tienda,Informática,Tablet 10,1,310.00,0
V2022-007,2022-08-09,C007,Laura Prieto,España,web,Almacenamiento,SSD 1TB,2,115.00,5
V2022-008,2022-09-25,C008,Pedro Santos,España,marketplace,Audio,Webcam HD,1,79.00,0
V2022-009,2022-10-14,C009,Sofía Ramos,Portugal,web,Informática,Portátil Air,1,780.00,8
V2022-010,2022-12-02,C010,Andrés Mora,España,tienda,Periféricos,Hub USB-C,4,35.00,0
"""
res3_1: 

In [4]:
// 2.3 ventas_2023.csv
val ventas2023 =
"""id_venta,fecha,id_cliente,cliente,pais,canal,categoria,producto,unidades,precio_unitario,descuento_pct
V2023-001,2023-01-09,C001,Ana García,España,web,Informática,Portátil Pro,1,970.00,4
V2023-002,2023-01-21,C011,Nuria Castro,España,marketplace,Audio,Auriculares Bluetooth,2,89.90,10
V2023-003,2023-02-15,C012,Raúl Gómez,Portugal,web,Monitores,Monitor 32,1,310.00,12
V2023-004,2023-03-05,C003,Marta López,España,tienda,Periféricos,Teclado Mecánico,1,79.00,0
V2023-005,2023-04-17,C013,Clara Soler,Francia,web,Informática,Tablet 10,2,299.00,5
V2023-006,2023-06-10,C014,David León,España,marketplace,Almacenamiento,Disco Externo 2TB,1,95.00,0
V2023-007,2023-07-19,C015,Lucía Torres,España,web,Audio,Micrófono USB,1,120.00,15
V2023-008,2023-09-01,C016,Iván Navarro,Portugal,tienda,Periféricos,Ratón Inalámbrico,2,31.00,5
V2023-009,2023-10-28,C017,Paula Marín,España,web,Informática,Portátil Air,1,810.00,7
V2023-010,2023-11-30,C018,Marcos Vidal,España,marketplace,Monitores,Monitor 27,2,215.00,10
"""

Files.write(Paths.get(s"$rutaBase/ventas_2023.csv"), ventas2023.getBytes(StandardCharsets.UTF_8))
println("✅ ventas_2023.csv creado")

✅ ventas_2023.csv creado


ventas2023: String = """id_venta,fecha,id_cliente,cliente,pais,canal,categoria,producto,unidades,precio_unitario,descuento_pct
V2023-001,2023-01-09,C001,Ana García,España,web,Informática,Portátil Pro,1,970.00,4
V2023-002,2023-01-21,C011,Nuria Castro,España,marketplace,Audio,Auriculares Bluetooth,2,89.90,10
V2023-003,2023-02-15,C012,Raúl Gómez,Portugal,web,Monitores,Monitor 32,1,310.00,12
V2023-004,2023-03-05,C003,Marta López,España,tienda,Periféricos,Teclado Mecánico,1,79.00,0
V2023-005,2023-04-17,C013,Clara Soler,Francia,web,Informática,Tablet 10,2,299.00,5
V2023-006,2023-06-10,C014,David León,España,marketplace,Almacenamiento,Disco Externo 2TB,1,95.00,0
V2023-007,2023-07-19,C015,Lucía Torres,España,web,Audio,Micrófono USB,1,120.00,15
V2023-008,2023-09-01,C016,Iván Navarro,Portugal,tienda,Periféricos,Ratón Inalámbrico,2,31.00,5
V2023-009,2023-10-28,C017,Paula Marín,España,web,Informática,Portátil Air,1,810.00,7
V2023-010,2023-11-30,C018,Marcos Vidal,España,marketplace,Monitores,Monito

In [5]:
// 2.4 ventas_2024.csv
val ventas2024 =
"""id_venta,fecha,id_cliente,cliente,pais,canal,categoria,producto,unidades,precio_unitario,descuento_pct
V2024-001,2024-01-12,C019,Isabel Romero,España,web,Informática,Portátil Pro,1,990.00,3
V2024-002,2024-02-08,C020,Hugo Molina,España,marketplace,Audio,Auriculares Bluetooth,1,94.90,5
V2024-003,2024-03-23,C021,Teresa Cano,Portugal,web,Almacenamiento,SSD 1TB,3,109.00,8
V2024-004,2024-04-04,C022,Álvaro Peña,España,tienda,Monitores,Monitor 32,1,330.00,10
V2024-005,2024-05-16,C023,Noelia Gil,Francia,web,Informática,Tablet 10,1,289.00,0
V2024-006,2024-06-27,C024,Rubén Flores,España,marketplace,Periféricos,Hub USB-C,2,39.00,0
V2024-007,2024-07-13,C025,Beatriz León,España,web,Audio,Micrófono USB,2,115.00,12
V2024-008,2024-08-29,C026,Sergio Vega,Portugal,tienda,Periféricos,Teclado Mecánico,1,82.00,0
V2024-009,2024-10-03,C027,Celia Robles,España,web,Informática,Portátil Air,1,835.00,6
V2024-010,2024-11-18,C028,Miguel Arias,España,marketplace,Monitores,Monitor 27,1,225.00,5
"""

Files.write(Paths.get(s"$rutaBase/ventas_2024.csv"), ventas2024.getBytes(StandardCharsets.UTF_8))
println("✅ ventas_2024.csv creado")

✅ ventas_2024.csv creado


ventas2024: String = """id_venta,fecha,id_cliente,cliente,pais,canal,categoria,producto,unidades,precio_unitario,descuento_pct
V2024-001,2024-01-12,C019,Isabel Romero,España,web,Informática,Portátil Pro,1,990.00,3
V2024-002,2024-02-08,C020,Hugo Molina,España,marketplace,Audio,Auriculares Bluetooth,1,94.90,5
V2024-003,2024-03-23,C021,Teresa Cano,Portugal,web,Almacenamiento,SSD 1TB,3,109.00,8
V2024-004,2024-04-04,C022,Álvaro Peña,España,tienda,Monitores,Monitor 32,1,330.00,10
V2024-005,2024-05-16,C023,Noelia Gil,Francia,web,Informática,Tablet 10,1,289.00,0
V2024-006,2024-06-27,C024,Rubén Flores,España,marketplace,Periféricos,Hub USB-C,2,39.00,0
V2024-007,2024-07-13,C025,Beatriz León,España,web,Audio,Micrófono USB,2,115.00,12
V2024-008,2024-08-29,C026,Sergio Vega,Portugal,tienda,Periféricos,Teclado Mecánico,1,82.00,0
V2024-009,2024-10-03,C027,Celia Robles,España,web,Informática,Portátil Air,1,835.00,6
V2024-010,2024-11-18,C028,Miguel Arias,España,marketplace,Monitores,Monitor 27,1,225.00,

---

## 📥 Parte 3 — Lectura de varios CSV como DataFrame

Spark puede leer una **carpeta entera** o un patrón glob `ventas_*.csv` con una sola instrucción y unirlos automáticamente.

In [6]:
val rutaCSV = s"$rutaBase/ventas_*.csv"

val dfRaw = spark.read
  .option("header", "true")
  .option("inferSchema", "true")
  .csv(rutaCSV)

println("=== Datos cargados desde varios CSV ===")
dfRaw.show(10, truncate = false)

println("=== Schema inferido ===")
dfRaw.printSchema()

println(s"Total de ventas cargadas: ${dfRaw.count()}")

=== Datos cargados desde varios CSV ===
+---------+----------+----------+------------+--------+-----------+--------------+---------------------+--------+---------------+-------------+
|id_venta |fecha     |id_cliente|cliente     |pais    |canal      |categoria     |producto             |unidades|precio_unitario|descuento_pct|
+---------+----------+----------+------------+--------+-----------+--------------+---------------------+--------+---------------+-------------+
|V2023-001|2023-01-09|C001      |Ana García  |España  |web        |Informática   |Portátil Pro         |1       |970.0          |4            |
|V2023-002|2023-01-21|C011      |Nuria Castro|España  |marketplace|Audio         |Auriculares Bluetooth|2       |89.9           |10           |
|V2023-003|2023-02-15|C012      |Raúl Gómez  |Portugal|web        |Monitores     |Monitor 32           |1       |310.0          |12           |
|V2023-004|2023-03-05|C003      |Marta López |España  |tienda     |Periféricos   |Teclado Mecáni

rutaCSV: String = "data/ventas_*.csv"
dfRaw: DataFrame = [id_venta: string, fecha: date ... 9 more fields]

**Preguntas de la Parte 3 — Respuestas**

1. **¿Cuántos CSV se han leído realmente?** → 3 (`ventas_2022.csv`, `ventas_2023.csv`, `ventas_2024.csv`).
2. **¿Spark ha unido los tres ficheros en un único DataFrame?** → Sí, porque comparten **mismo header** y mismo orden de columnas. El glob `ventas_*.csv` los concatena de forma transparente.
3. **¿Qué tipos ha inferido Spark?** → `fecha` como `string` (no como `date`, lo veremos en la Parte 4), `unidades` como `int`, `precio_unitario` y `descuento_pct` como `double`.
4. **¿Por qué no conviene depender de `inferSchema` en producción?** → Porque (a) obliga a Spark a leer dos veces el fichero (una para inferir, otra para cargar) — costoso con GB de datos; (b) los tipos pueden cambiar entre ejecuciones si llegan filas con formatos distintos; (c) las fechas casi nunca se infieren correctamente; (d) un schema explícito es **autodocumentado** y previene errores silenciosos.

---

## 🧹 Parte 4 — Normalización de tipos y columnas

Antes de convertir a `Dataset` conviene controlar los tipos y limpiar texto. Las funciones nativas de Spark son eficientes y optimizables por Catalyst.

In [7]:
val dfNormalizado = dfRaw
  .withColumn("fecha", to_date(col("fecha"), "yyyy-MM-dd"))
  .withColumn("unidades", col("unidades").cast(IntegerType))
  .withColumn("precio_unitario", col("precio_unitario").cast(DoubleType))
  .withColumn("descuento_pct", col("descuento_pct").cast(DoubleType))
  .withColumn("pais", trim(col("pais")))
  .withColumn("canal", lower(trim(col("canal"))))
  .withColumn("categoria", trim(col("categoria")))
  .withColumn("producto", trim(col("producto")))

println("=== DataFrame normalizado ===")
dfNormalizado.show(10, truncate = false)

dfNormalizado.printSchema()

=== DataFrame normalizado ===
+---------+----------+----------+------------+--------+-----------+--------------+---------------------+--------+---------------+-------------+
|id_venta |fecha     |id_cliente|cliente     |pais    |canal      |categoria     |producto             |unidades|precio_unitario|descuento_pct|
+---------+----------+----------+------------+--------+-----------+--------------+---------------------+--------+---------------+-------------+
|V2023-001|2023-01-09|C001      |Ana García  |España  |web        |Informática   |Portátil Pro         |1       |970.0          |4.0          |
|V2023-002|2023-01-21|C011      |Nuria Castro|España  |marketplace|Audio         |Auriculares Bluetooth|2       |89.9           |10.0         |
|V2023-003|2023-02-15|C012      |Raúl Gómez  |Portugal|web        |Monitores     |Monitor 32           |1       |310.0          |12.0         |
|V2023-004|2023-03-05|C003      |Marta López |España  |tienda     |Periféricos   |Teclado Mecánico     |1 

dfNormalizado: DataFrame = [id_venta: string, fecha: date ... 9 more fields]

---

## 🎯 Parte 5 — Selección de columnas útiles

Solo seleccionamos las columnas relevantes para la lógica de negocio del Dataset.

In [8]:
val dfVentasSeleccionadas = dfNormalizado.select(
  col("id_venta"),
  col("fecha"),
  col("id_cliente"),
  col("cliente"),
  col("pais"),
  col("canal"),
  col("categoria"),
  col("producto"),
  col("unidades"),
  col("precio_unitario"),
  col("descuento_pct")
)

println("=== Columnas seleccionadas ===")
dfVentasSeleccionadas.show(5, truncate = false)

=== Columnas seleccionadas ===
+---------+----------+----------+------------+--------+-----------+-----------+---------------------+--------+---------------+-------------+
|id_venta |fecha     |id_cliente|cliente     |pais    |canal      |categoria  |producto             |unidades|precio_unitario|descuento_pct|
+---------+----------+----------+------------+--------+-----------+-----------+---------------------+--------+---------------+-------------+
|V2023-001|2023-01-09|C001      |Ana García  |España  |web        |Informática|Portátil Pro         |1       |970.0          |4.0          |
|V2023-002|2023-01-21|C011      |Nuria Castro|España  |marketplace|Audio      |Auriculares Bluetooth|2       |89.9           |10.0         |
|V2023-003|2023-02-15|C012      |Raúl Gómez  |Portugal|web        |Monitores  |Monitor 32           |1       |310.0          |12.0         |
|V2023-004|2023-03-05|C003      |Marta López |España  |tienda     |Periféricos|Teclado Mecánico     |1       |79.0         

dfVentasSeleccionadas: DataFrame = [id_venta: string, fecha: date ... 9 more fields]

---

## 🧬 Parte 6 — Convertir DataFrame a Dataset tipado

### 6.1 Definir la `case class` de entrada

In [9]:
import java.sql.Date

case class VentaRaw(
  id_venta: String,
  fecha: Date,
  id_cliente: String,
  cliente: String,
  pais: String,
  canal: String,
  categoria: String,
  producto: String,
  unidades: Int,
  precio_unitario: Double,
  descuento_pct: Double
)

println("✅ case class VentaRaw definida")

✅ case class VentaRaw definida


import java.sql.Date
defined class VentaRaw

### 6.2 Convertir el DataFrame a `Dataset[VentaRaw]`

In [10]:
val dsVentas: Dataset[VentaRaw] = dfVentasSeleccionadas.as[VentaRaw]

println("=== Dataset[VentaRaw] ===")
dsVentas.show(5, truncate = false)

println(s"Tipo: ${dsVentas.getClass.getSimpleName}")
println(s"Total registros: ${dsVentas.count()}")

=== Dataset[VentaRaw] ===
+---------+----------+----------+------------+--------+-----------+-----------+---------------------+--------+---------------+-------------+
|id_venta |fecha     |id_cliente|cliente     |pais    |canal      |categoria  |producto             |unidades|precio_unitario|descuento_pct|
+---------+----------+----------+------------+--------+-----------+-----------+---------------------+--------+---------------+-------------+
|V2023-001|2023-01-09|C001      |Ana García  |España  |web        |Informática|Portátil Pro         |1       |970.0          |4.0          |
|V2023-002|2023-01-21|C011      |Nuria Castro|España  |marketplace|Audio      |Auriculares Bluetooth|2       |89.9           |10.0         |
|V2023-003|2023-02-15|C012      |Raúl Gómez  |Portugal|web        |Monitores  |Monitor 32           |1       |310.0          |12.0         |
|V2023-004|2023-03-05|C003      |Marta López |España  |tienda     |Periféricos|Teclado Mecánico     |1       |79.0           |0.

dsVentas: Dataset[VentaRaw] = [id_venta: string, fecha: date ... 9 more fields]

---

## 💼 Parte 7 — Lógica de negocio segura con Scala

### 7.1 Definir `case class VentaEnriquecida`

In [11]:
case class VentaEnriquecida(
  id_venta: String,
  fecha: Date,
  id_cliente: String,
  cliente: String,
  pais: String,
  canal: String,
  categoria: String,
  producto: String,
  unidades: Int,
  precio_unitario: Double,
  descuento_pct: Double,
  importe_bruto: Double,
  importe_descuento: Double,
  importe_neto: Double,
  segmento_venta: String,
  requiere_revision: Boolean
)

println("✅ case class VentaEnriquecida definida")

✅ case class VentaEnriquecida definida


defined class VentaEnriquecida

### 7.2 Funciones Scala de negocio

| Regla | Resultado |
| --- | --- |
| Importe neto >= 900 | `Venta estratégica` |
| Importe neto >= 300 | `Venta media` |
| Importe neto < 300 | `Venta pequeña` |
| Descuento > 12% **y** importe neto > 200 | `requiere_revision = true` |

In [12]:
def clasificarVenta(importeNeto: Double): String = {
  if      (importeNeto >= 900) "Venta estratégica"
  else if (importeNeto >= 300) "Venta media"
  else                         "Venta pequeña"
}

def necesitaRevision(descuentoPct: Double, importeNeto: Double): Boolean = {
  descuentoPct > 12.0 && importeNeto > 200.0
}

println("✅ Funciones de negocio definidas: clasificarVenta, necesitaRevision")

✅ Funciones de negocio definidas: clasificarVenta, necesitaRevision


defined function clasificarVenta
defined function necesitaRevision

### 7.3 Aplicar la lógica sobre el Dataset con `map`

In [13]:
val dsEnriquecido: Dataset[VentaEnriquecida] = dsVentas.map { v =>
  val importeBruto     = v.unidades * v.precio_unitario
  val importeDescuento = importeBruto * (v.descuento_pct / 100.0)
  val importeNeto      = importeBruto - importeDescuento

  VentaEnriquecida(
    id_venta          = v.id_venta,
    fecha             = v.fecha,
    id_cliente        = v.id_cliente,
    cliente           = v.cliente,
    pais              = v.pais,
    canal             = v.canal,
    categoria         = v.categoria,
    producto          = v.producto,
    unidades          = v.unidades,
    precio_unitario   = v.precio_unitario,
    descuento_pct     = v.descuento_pct,
    importe_bruto     = BigDecimal(importeBruto).setScale(2, BigDecimal.RoundingMode.HALF_UP).toDouble,
    importe_descuento = BigDecimal(importeDescuento).setScale(2, BigDecimal.RoundingMode.HALF_UP).toDouble,
    importe_neto      = BigDecimal(importeNeto).setScale(2, BigDecimal.RoundingMode.HALF_UP).toDouble,
    segmento_venta    = clasificarVenta(importeNeto),
    requiere_revision = necesitaRevision(v.descuento_pct, importeNeto)
  )
}

println("=== Dataset enriquecido con lógica Scala ===")
dsEnriquecido.show(10, truncate = false)

=== Dataset enriquecido con lógica Scala ===
+---------+----------+----------+------------+--------+-----------+--------------+---------------------+--------+---------------+-------------+-------------+-----------------+------------+-----------------+-----------------+
|id_venta |fecha     |id_cliente|cliente     |pais    |canal      |categoria     |producto             |unidades|precio_unitario|descuento_pct|importe_bruto|importe_descuento|importe_neto|segmento_venta   |requiere_revision|
+---------+----------+----------+------------+--------+-----------+--------------+---------------------+--------+---------------+-------------+-------------+-----------------+------------+-----------------+-----------------+
|V2023-001|2023-01-09|C001      |Ana García  |España  |web        |Informática   |Portátil Pro         |1       |970.0          |4.0          |970.0        |38.8             |931.2       |Venta estratégica|false            |
|V2023-002|2023-01-21|C011      |Nuria Castro|España  |

dsEnriquecido: Dataset[VentaEnriquecida] = [id_venta: string, fecha: date ... 14 more fields]

**Preguntas Parte 7 — Respuestas**

1. **¿Por qué Dataset y no DataFrame aquí?** → Porque la lógica `clasificarVenta` y `necesitaRevision` está escrita en **Scala puro** y se beneficia de la verificación de tipos en compilación. Con DataFrame perderíamos los tipos (todo sería `Row`/`Any`) y los errores aparecerían en runtime.
2. **¿Qué aporta `case class VentaRaw`?** → Acceso a los campos como propiedades Scala (`v.precio_unitario`), serialización automática con encoders de Spark y autocompletado en el IDE.
3. **¿Qué pasaría si escribimos `v.precio_unitarioo`?** → **Error de compilación** (no compila el notebook). Con DataFrame habría que esperar al runtime para descubrir el typo.

---

## 🔄 Parte 8 — Convertir Dataset enriquecido a DataFrame

Para escribir, particionar y consultar con Spark SQL volvemos a la API de DataFrame.

In [14]:
val dfEnriquecido: DataFrame = dsEnriquecido.toDF()

println("=== DataFrame enriquecido ===")
dfEnriquecido.show(5, truncate = false)
dfEnriquecido.printSchema()

=== DataFrame enriquecido ===
+---------+----------+----------+------------+--------+-----------+-----------+---------------------+--------+---------------+-------------+-------------+-----------------+------------+-----------------+-----------------+
|id_venta |fecha     |id_cliente|cliente     |pais    |canal      |categoria  |producto             |unidades|precio_unitario|descuento_pct|importe_bruto|importe_descuento|importe_neto|segmento_venta   |requiere_revision|
+---------+----------+----------+------------+--------+-----------+-----------+---------------------+--------+---------------+-------------+-------------+-----------------+------------+-----------------+-----------------+
|V2023-001|2023-01-09|C001      |Ana García  |España  |web        |Informática|Portátil Pro         |1       |970.0          |4.0          |970.0        |38.8             |931.2       |Venta estratégica|false            |
|V2023-002|2023-01-21|C011      |Nuria Castro|España  |marketplace|Audio      |Aur

dfEnriquecido: DataFrame = [id_venta: string, fecha: date ... 14 more fields]

---

## 🛡️ Parte 9 — UDF para riesgo comercial

### 9.1 Reglas de negocio

| Condición | Código de riesgo |
| --- | --- |
| Marketplace + descuento ≥ 10 + importe neto > 300 | `RIESGO_MARKETPLACE_DESCUENTO` |
| País ≠ España + importe neto > 500 | `RIESGO_INTERNACIONAL_ALTO` |
| Categoría Informática + importe neto > 800 | `VENTA_TECNOLOGICA_CLAVE` |
| Resto | `NORMAL` |

In [15]:
// 9.2 Crear y registrar la UDF
val clasificarRiesgoUDF = udf {
  (pais: String, canal: String, categoria: String, descuento: Double, importeNeto: Double) =>
    val canalNorm     = Option(canal).map(_.toLowerCase).getOrElse("")
    val paisNorm      = Option(pais).getOrElse("")
    val categoriaNorm = Option(categoria).getOrElse("")

    if (canalNorm == "marketplace" && descuento >= 10.0 && importeNeto > 300.0)
      "RIESGO_MARKETPLACE_DESCUENTO"
    else if (paisNorm != "España" && importeNeto > 500.0)
      "RIESGO_INTERNACIONAL_ALTO"
    else if (categoriaNorm == "Informática" && importeNeto > 800.0)
      "VENTA_TECNOLOGICA_CLAVE"
    else
      "NORMAL"
}

spark.udf.register("clasificar_riesgo", clasificarRiesgoUDF)

println("✅ UDF clasificarRiesgoUDF registrada")

✅ UDF clasificarRiesgoUDF registrada


clasificarRiesgoUDF: org.apache.spark.sql.expressions.UserDefinedFunction = SparkUserDefinedFunction(
  f = ammonite.$sess.cmd15$Helper$$Lambda$7755/0x0000020d028a5970@621dde29,
  dataType = StringType,
  inputEncoders = ArraySeq(
    Some(value = StringEncoder),
    Some(value = StringEncoder),
    Some(value = StringEncoder),
    Some(value = PrimitiveDoubleEncoder),
    Some(value = PrimitiveDoubleEncoder)
  ),
  outputEncoder = Some(value = StringEncoder),
  givenName = None,
  nullable = true,
  deterministic = true
)
res15_1: org.apache.spark.sql.expressions.UserDefinedFunction = SparkUserDefinedFunction(
  f = ammonite.$sess.cmd15$Helper$$Lambda$7755/0x0000020d028a5970@621dde29,
  dataType = StringType,
  inputEncoders = ArraySeq(
    Some(value = StringEncoder),
    Some(value = StringEncoder),
    Some(value = StringEncoder),
    Some(value = PrimitiveDoubleEncoder),
    Some(value = PrimitiveDoubleEncoder)
  ),
  outputEncoder = Some(value = StringEncoder),
  givenName = Some

In [16]:
// 9.3 Aplicar la UDF
val dfConRiesgo = dfEnriquecido
  .withColumn(
    "riesgo_comercial",
    clasificarRiesgoUDF(
      col("pais"),
      col("canal"),
      col("categoria"),
      col("descuento_pct"),
      col("importe_neto")
    )
  )

println("=== DataFrame con riesgo comercial ===")
dfConRiesgo
  .select("id_venta", "pais", "canal", "categoria", "descuento_pct", "importe_neto", "riesgo_comercial")
  .show(30, truncate = false)

println("\n=== Distribución del riesgo ===")
dfConRiesgo.groupBy("riesgo_comercial").count().orderBy($"count".desc).show(false)

=== DataFrame con riesgo comercial ===
+---------+--------+-----------+--------------+-------------+------------+----------------------------+
|id_venta |pais    |canal      |categoria     |descuento_pct|importe_neto|riesgo_comercial            |
+---------+--------+-----------+--------------+-------------+------------+----------------------------+
|V2023-001|España  |web        |Informática   |4.0          |931.2       |VENTA_TECNOLOGICA_CLAVE     |
|V2023-002|España  |marketplace|Audio         |10.0         |161.82      |NORMAL                      |
|V2023-003|Portugal|web        |Monitores     |12.0         |272.8       |NORMAL                      |
|V2023-004|España  |tienda     |Periféricos   |0.0          |79.0        |NORMAL                      |
|V2023-005|Francia |web        |Informática   |5.0          |568.1       |RIESGO_INTERNACIONAL_ALTO   |
|V2023-006|España  |marketplace|Almacenamiento|0.0          |95.0        |NORMAL                      |
|V2023-007|España  |web  

dfConRiesgo: DataFrame = [id_venta: string, fecha: date ... 15 more fields]

> 💡 **Nota didáctica:** usamos UDF solo cuando la lógica no se expresa cómodamente con funciones nativas. Si pudiera hacerse con `when`/`col`/`concat`/`year`/`lower`/`regexp_replace`… siempre sería preferible, ya que Catalyst optimiza las funciones nativas pero **no** puede inspeccionar el cuerpo de una UDF.

---

## ➕ Parte 10 — Columnas derivadas con funciones nativas Spark SQL

In [17]:
val dfFinal = dfConRiesgo
  .withColumn("anio", year(col("fecha")))
  .withColumn("mes", month(col("fecha")))
  .withColumn("trimestre", quarter(col("fecha")))
  .withColumn("importe_neto_redondeado", round(col("importe_neto"), 2))
  .withColumn(
    "tipo_cliente",
    when(col("importe_neto") >= 900, "premium")
      .when(col("importe_neto") >= 300, "estandar")
      .otherwise("ocasional")
  )
  .withColumn("venta_internacional", when(col("pais") =!= "España", true).otherwise(false))

println("=== DataFrame final con columnas nativas Spark SQL ===")
dfFinal.show(10, truncate = false)

=== DataFrame final con columnas nativas Spark SQL ===
+---------+----------+----------+------------+--------+-----------+--------------+---------------------+--------+---------------+-------------+-------------+-----------------+------------+-----------------+-----------------+----------------------------+----+---+---------+-----------------------+------------+-------------------+
|id_venta |fecha     |id_cliente|cliente     |pais    |canal      |categoria     |producto             |unidades|precio_unitario|descuento_pct|importe_bruto|importe_descuento|importe_neto|segmento_venta   |requiere_revision|riesgo_comercial            |anio|mes|trimestre|importe_neto_redondeado|tipo_cliente|venta_internacional|
+---------+----------+----------+------------+--------+-----------+--------------+---------------------+--------+---------------+-------------+-------------+-----------------+------------+-----------------+-----------------+----------------------------+----+---+---------+-------------

dfFinal: DataFrame = [id_venta: string, fecha: date ... 21 more fields]

---

## 🎁 Parte 11 — Capa analítica final (selección reducida)

In [18]:
val dfCapaAnalitica = dfFinal.select(
  col("id_venta"),
  col("fecha"),
  col("anio"),
  col("mes"),
  col("trimestre"),
  col("id_cliente"),
  col("pais"),
  col("canal"),
  col("categoria"),
  col("producto"),
  col("unidades"),
  col("importe_bruto"),
  col("importe_descuento"),
  col("importe_neto_redondeado").as("importe_neto"),
  col("segmento_venta"),
  col("tipo_cliente"),
  col("venta_internacional"),
  col("requiere_revision"),
  col("riesgo_comercial")
)

println("=== Capa analítica final ===")
dfCapaAnalitica.show(10, truncate = false)

dfCapaAnalitica.printSchema()
println(s"Total filas: ${dfCapaAnalitica.count()}")

=== Capa analítica final ===
+---------+----------+----+---+---------+----------+--------+-----------+--------------+---------------------+--------+-------------+-----------------+------------+-----------------+------------+-------------------+-----------------+----------------------------+
|id_venta |fecha     |anio|mes|trimestre|id_cliente|pais    |canal      |categoria     |producto             |unidades|importe_bruto|importe_descuento|importe_neto|segmento_venta   |tipo_cliente|venta_internacional|requiere_revision|riesgo_comercial            |
+---------+----------+----+---+---------+----------+--------+-----------+--------------+---------------------+--------+-------------+-----------------+------------+-----------------+------------+-------------------+-----------------+----------------------------+
|V2023-001|2023-01-09|2023|1  |1        |C001      |España  |web        |Informática   |Portátil Pro         |1       |970.0        |38.8             |931.2       |Venta estratégica|

dfCapaAnalitica: DataFrame = [id_venta: string, fecha: date ... 17 more fields]

---

## 💾 Parte 12 — Guardar en Parquet particionado por `anio` y `mes`

Generará una estructura tipo:

```text
parquet_final_ventas/
  anio=2022/
    mes=1/
    mes=2/
    ...
  anio=2023/
  anio=2024/
```

In [19]:
val rutaParquetFinal = s"$rutaBase/salida/parquet_final_ventas"

dfCapaAnalitica.write
  .mode("overwrite")
  .partitionBy("anio", "mes")
  .parquet(rutaParquetFinal)

println(s"✅ Parquet final particionado escrito en: $rutaParquetFinal")

// Mostrar las particiones generadas
println("\nParticiones de primer nivel (anio=...):")
new File(rutaParquetFinal)
  .listFiles()
  .filter(_.isDirectory)
  .map(_.getName)
  .sorted
  .foreach(p => println(s"  $p/"))

✅ Parquet final particionado escrito en: data/salida/parquet_final_ventas

Particiones de primer nivel (anio=...):
  anio=2022/
  anio=2023/
  anio=2024/


rutaParquetFinal: String = "data/salida/parquet_final_ventas"

---

## 📖 Parte 13 — Leer el Parquet final

In [20]:
val dfParquetFinal = spark.read.parquet(rutaParquetFinal)

println("=== Parquet final leído ===")
dfParquetFinal.show(10, truncate = false)

dfParquetFinal.printSchema()
println(s"Total filas: ${dfParquetFinal.count()}")

=== Parquet final leído ===
+---------+----------+---------+----------+--------+-----------+--------------+---------------------+--------+-------------+-----------------+------------+-----------------+------------+-------------------+-----------------+----------------------------+----+---+
|id_venta |fecha     |trimestre|id_cliente|pais    |canal      |categoria     |producto             |unidades|importe_bruto|importe_descuento|importe_neto|segmento_venta   |tipo_cliente|venta_internacional|requiere_revision|riesgo_comercial            |anio|mes|
+---------+----------+---------+----------+--------+-----------+--------------+---------------------+--------+-------------+-----------------+------------+-----------------+------------+-------------------+-----------------+----------------------------+----+---+
|V2023-010|2023-11-30|4        |C018      |España  |marketplace|Monitores     |Monitor 27           |2       |430.0        |43.0             |387.0       |Venta media      |estandar  

dfParquetFinal: DataFrame = [id_venta: string, fecha: date ... 17 more fields]

---

## 🔍 Parte 14 — Consultas con Spark SQL

### 14.1 Crear vista temporal

In [21]:
dfParquetFinal.createOrReplaceTempView("ventas_retailnova")

println("✅ Vista 'ventas_retailnova' registrada")
val totalFilas = spark.sql("SELECT COUNT(*) AS n FROM ventas_retailnova").collect()(0).getLong(0)
println(s"   Filas en la vista: $totalFilas")

✅ Vista 'ventas_retailnova' registrada
   Filas en la vista: 30


totalFilas: Long = 30L

### 14.2 Consulta 1 — Ventas por año y mes

In [22]:
spark.sql("""
  SELECT
    anio,
    mes,
    COUNT(*)                    AS total_ventas,
    ROUND(SUM(importe_neto), 2) AS facturacion_neta
  FROM ventas_retailnova
  GROUP BY anio, mes
  ORDER BY anio, mes
""").show(100, truncate = false)

+----+---+------------+----------------+
|anio|mes|total_ventas|facturacion_neta|
+----+---+------------+----------------+
|2022|1  |1           |902.5           |
|2022|2  |1           |150.0           |
|2022|3  |1           |80.73           |
|2022|4  |1           |113.98          |
|2022|5  |1           |187.0           |
|2022|6  |1           |310.0           |
|2022|8  |1           |218.5           |
|2022|9  |1           |79.0            |
|2022|10 |1           |717.6           |
|2022|12 |1           |140.0           |
|2023|1  |2           |1093.02         |
|2023|2  |1           |272.8           |
|2023|3  |1           |79.0            |
|2023|4  |1           |568.1           |
|2023|6  |1           |95.0            |
|2023|7  |1           |102.0           |
|2023|9  |1           |58.9            |
|2023|10 |1           |753.3           |
|2023|11 |1           |387.0           |
|2024|1  |1           |960.3           |
|2024|2  |1           |90.16           |
|2024|3  |1     

### 14.3 Consulta 2 — Facturación por país

In [23]:
spark.sql("""
  SELECT
    pais,
    COUNT(*)                    AS total_ventas,
    ROUND(SUM(importe_neto), 2) AS facturacion_neta,
    ROUND(AVG(importe_neto), 2) AS ticket_medio
  FROM ventas_retailnova
  GROUP BY pais
  ORDER BY facturacion_neta DESC
""").show(false)

+--------+------------+----------------+------------+
|pais    |total_ventas|facturacion_neta|ticket_medio|
+--------+------------+----------------+------------+
|España  |21          |6893.56         |328.26      |
|Portugal|6           |1546.12         |257.69      |
|Francia |3           |1167.1          |389.03      |
+--------+------------+----------------+------------+



### 14.4 Consulta 3 — Ventas que requieren revisión

In [24]:
spark.sql("""
  SELECT
    id_venta,
    fecha,
    pais,
    canal,
    categoria,
    producto,
    importe_neto,
    descuento_pct,
    requiere_revision,
    riesgo_comercial
  FROM ventas_retailnova
  WHERE requiere_revision = true
     OR riesgo_comercial <> 'NORMAL'
  ORDER BY importe_neto DESC
""").show(100, truncate = false)

org.apache.spark.sql.catalyst.ExtendedAnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `descuento_pct` cannot be resolved. Did you mean one of the following? [`id_venta`, `segmento_venta`, `anio`, `canal`, `categoria`]. SQLSTATE: 42703; line 10 pos 4;
'Sort ['importe_neto DESC NULLS LAST], true
+- 'Project [id_venta#791, fecha#792, pais#795, canal#796, categoria#797, producto#798, importe_neto#802, 'descuento_pct, requiere_revision#806, riesgo_comercial#807]
   +- Filter ((requiere_revision#806 = true) OR NOT (riesgo_comercial#807 = NORMAL))
      +- SubqueryAlias ventas_retailnova
         +- View (`ventas_retailnova`, [id_venta#791, fecha#792, trimestre#793, id_cliente#794, pais#795, canal#796, categoria#797, producto#798, unidades#799, importe_bruto#800, importe_descuento#801, importe_neto#802, segmento_venta#803, tipo_cliente#804, venta_internacional#805, requiere_revision#806, riesgo_comercial#807, anio#808, mes#809])
     

> ⚠️ La columna `descuento_pct` no se ha incluido explícitamente en la capa analítica. Si la consulta anterior falla por columna no encontrada, vuelve al paso 11 y añade `col("descuento_pct")` al `select` o reescribe la query sin ese campo.

### 14.5 Consulta 4 — Ranking de categorías

In [25]:
spark.sql("""
  SELECT
    categoria,
    COUNT(*)                    AS total_ventas,
    ROUND(SUM(importe_neto), 2) AS facturacion_neta
  FROM ventas_retailnova
  GROUP BY categoria
  ORDER BY facturacion_neta DESC
""").show(false)

+--------------+------------+----------------+
|categoria     |total_ventas|facturacion_neta|
+--------------+------------+----------------+
|Informática   |9           |6216.9          |
|Monitores     |5           |1357.55         |
|Audio         |6           |749.36          |
|Periféricos   |7           |668.63          |
|Almacenamiento|3           |614.34          |
+--------------+------------+----------------+



---

## ⏱️ Parte 15 — Benchmark Avro / ORC / Parquet

### 15.1 Funciones auxiliares

In [27]:
def medirTiempo[T](bloque: => T): (T, Long) = {
  val inicio = System.nanoTime()
  val resultado = bloque
  val fin = System.nanoTime()
  val tiempoMs = (fin - inicio) / 1000000L
  (resultado, tiempoMs)
}

def calcularTamanoBytes(path: String): Long = {
  val p = Paths.get(path)
  if (!Files.exists(p)) 0L
  else {
    Files.walk(p)
      .filter(Files.isRegularFile(_))
      .mapToLong(Files.size(_))
      .sum()
  }
}

println("✅ Funciones medirTiempo y calcularTamanoBytes definidas")

✅ Funciones medirTiempo y calcularTamanoBytes definidas


defined function medirTiempo
defined function calcularTamanoBytes

### 15.2 Escritura en los tres formatos

In [28]:
val rutaBenchParquet = s"$rutaBase/benchmark/ventas_parquet"
val rutaBenchORC     = s"$rutaBase/benchmark/ventas_orc"
val rutaBenchAvro    = s"$rutaBase/benchmark/ventas_avro"

val (_, tEscParquet) = medirTiempo {
  dfCapaAnalitica.write.mode("overwrite").parquet(rutaBenchParquet)
}

val (_, tEscORC) = medirTiempo {
  dfCapaAnalitica.write.mode("overwrite").orc(rutaBenchORC)
}

val (_, tEscAvro) = medirTiempo {
  dfCapaAnalitica.write.mode("overwrite").format("avro").save(rutaBenchAvro)
}

println(s"⏱️ Escritura Parquet: $tEscParquet ms")
println(s"⏱️ Escritura ORC    : $tEscORC ms")
println(s"⏱️ Escritura Avro   : $tEscAvro ms")

⏱️ Escritura Parquet: 611 ms
⏱️ Escritura ORC    : 585 ms
⏱️ Escritura Avro   : 636 ms


rutaBenchParquet: String = "data/benchmark/ventas_parquet"
rutaBenchORC: String = "data/benchmark/ventas_orc"
rutaBenchAvro: String = "data/benchmark/ventas_avro"
tEscParquet: Long = 611L
tEscORC: Long = 585L
tEscAvro: Long = 636L

### 15.3 Lectura completa con `count()`

In [29]:
val (_, tLecParquet) = medirTiempo { spark.read.parquet(rutaBenchParquet).count() }
val (_, tLecORC)     = medirTiempo { spark.read.orc(rutaBenchORC).count() }
val (_, tLecAvro)    = medirTiempo { spark.read.format("avro").load(rutaBenchAvro).count() }

println(s"📖 Lectura Parquet + count: $tLecParquet ms")
println(s"📖 Lectura ORC + count    : $tLecORC ms")
println(s"📖 Lectura Avro + count   : $tLecAvro ms")

📖 Lectura Parquet + count: 339 ms
📖 Lectura ORC + count    : 310 ms
📖 Lectura Avro + count   : 233 ms


tLecParquet: Long = 339L
tLecORC: Long = 310L
tLecAvro: Long = 233L

### 15.4 Tabla comparativa

In [30]:
val tamParquet = calcularTamanoBytes(rutaBenchParquet)
val tamORC     = calcularTamanoBytes(rutaBenchORC)
val tamAvro    = calcularTamanoBytes(rutaBenchAvro)

val comparativa = Seq(
  ("Parquet", tEscParquet, tLecParquet, tamParquet),
  ("ORC",     tEscORC,     tLecORC,     tamORC),
  ("Avro",    tEscAvro,    tLecAvro,    tamAvro)
).toDF("formato", "tiempo_escritura_ms", "tiempo_lectura_ms", "tamano_bytes")

println("=== Comparativa Avro / ORC / Parquet ===")
comparativa.show(false)

=== Comparativa Avro / ORC / Parquet ===
+-------+-------------------+-----------------+------------+
|formato|tiempo_escritura_ms|tiempo_lectura_ms|tamano_bytes|
+-------+-------------------+-----------------+------------+
|Parquet|611                |339              |19041       |
|ORC    |585                |310              |9559        |
|Avro   |636                |233              |5507        |
+-------+-------------------+-----------------+------------+



tamParquet: Long = 19041L
tamORC: Long = 9559L
tamAvro: Long = 5507L
comparativa: DataFrame = [formato: string, tiempo_escritura_ms: bigint ... 2 more fields]

---

## 🎓 Parte 16 — Preguntas finales

### Preguntas técnicas

1. **¿Por qué se leen los CSV primero como DataFrame?** → Porque la API de DataFrame es la más cómoda para lectura masiva con globs (`ventas_*.csv`), inferencia/aplicación de schema y operaciones columnares optimizables por Catalyst.
2. **¿Por qué normalizamos tipos antes de convertir a Dataset?** → Porque la `case class VentaRaw` exige que cada campo tenga el tipo exacto declarado (`Date`, `Int`, `Double`). Si los tipos no coinciden, `as[VentaRaw]` falla en runtime.
3. **¿Qué aporta `case class VentaRaw`?** → Tipado seguro, acceso a campos como propiedades Scala, encoders automáticos.
4. **¿Qué aporta `case class VentaEnriquecida`?** → Documenta el contrato de salida del paso de negocio y permite que el `map` retorne un `Dataset[VentaEnriquecida]` también tipado.
5. **¿Por qué volvemos de Dataset a DataFrame?** → Spark SQL, particionado, vistas temporales y la mayoría de optimizaciones funcionan de forma más natural sobre DataFrames.
6. **¿Por qué guardamos en Parquet y no en CSV?** → Schema embebido, compresión columnar, lectura selectiva de columnas, predicado *pushdown* y estadísticas por bloque.
7. **¿Qué ventaja tiene particionar por `anio` y `mes`?** → Cuando una consulta filtra por esos campos, Spark hace **partition pruning**: ignora carpetas enteras y solo lee los datos relevantes.
8. **¿Qué ocurre si una consulta filtra por `anio = 2024`?** → Spark lee únicamente la subcarpeta `anio=2024/`, descartando el resto sin tocarlo. 


---
### Preguntas de negocio

1. ¿Qué país genera más facturación? (Consulta 14.3)
2. ¿Qué categoría tiene mayor facturación neta? (Consulta 14.5)
3. ¿Qué ventas deberían ser revisadas por el equipo comercial? (Consulta 14.4)
4. ¿Qué canal genera más ventas estratégicas?
5. ¿Qué meses tienen mayor volumen de ventas? (Consulta 14.2)


### ✅ Respuestas de negocio

**1. ¿Qué país genera más facturación?**

🇪🇸 **España** lidera con diferencia: **≈ 6 893,56 €** de facturación neta acumulada en los 3 años, frente a **≈ 1 546,12 €** de Portugal y **≈ 1 167,10 €** de Francia. España concentra aproximadamente el **71,8 %** del negocio total.

**2. ¿Qué categoría tiene mayor facturación neta?**

💻 **Informática** es la categoría dominante con **≈ 6 216,90 €**, muy por encima del resto:

| Categoría | Facturación neta |
| --- | --- |
| Informática | ≈ 6 216,90 € |
| Monitores | ≈ 1 357,55 € |
| Audio | ≈ 749,36 € |
| Periféricos | ≈ 668,63 € |
| Almacenamiento | ≈ 614,34 € |

Los portátiles (`Portátil Pro` y `Portátil Air`) son los productos que tiran del negocio.

**3. ¿Qué ventas deberían ser revisadas por el equipo comercial?**

Ninguna venta cumple `requiere_revision = true` (la regla exige descuento **estricto** > 12 % **y** importe neto > 200 €; los descuentos de 12 % no la disparan, y los > 12 % se quedan por debajo de 200 € de neto).

Sin embargo, **6 ventas** activan `riesgo_comercial` y deberían pasar revisión:

| id_venta | país | canal | categoría | importe neto | riesgo |
| --- | --- | --- | --- | --- | --- |
| V2024-001 | España | web | Informática | 960,30 € | VENTA_TECNOLOGICA_CLAVE |
| V2023-001 | España | web | Informática | 931,20 € | VENTA_TECNOLOGICA_CLAVE |
| V2022-001 | España | web | Informática | 902,50 € | VENTA_TECNOLOGICA_CLAVE |
| V2022-009 | Portugal | web | Informática | 717,60 € | RIESGO_INTERNACIONAL_ALTO |
| V2023-005 | Francia | web | Informática | 568,10 € | RIESGO_INTERNACIONAL_ALTO |
| V2023-010 | España | marketplace | Monitores | 387,00 € | RIESGO_MARKETPLACE_DESCUENTO |

**4. ¿Qué canal genera más ventas estratégicas?**

Solo **3 ventas** alcanzan el umbral de *Venta estratégica* (neto ≥ 900 €): `V2022-001`, `V2023-001` y `V2024-001`. **Las tres se realizaron por el canal `web`**, así que el canal estrella es **`web`** (100 % de las estratégicas). El `marketplace` no genera ninguna venta estratégica en estos 3 años.

**5. ¿Qué meses tienen mayor volumen de ventas?**

📅 **Enero** es claramente el mes más fuerte, tanto en número de operaciones como en facturación:

| Mes | Nº ventas (3 años) | Facturación neta acumulada |
| --- | --- | --- |
| **01 — Enero** | **4** | **≈ 2 955,82 €** |
| 02 / 03 / 04 / 06 / 10 | 3 cada uno | volúmenes intermedios |
| 05 / 07 / 08 / 09 / 11 | 2 cada uno | bajo |
| 12 — Diciembre | 1 | el más bajo |

---

### 🧭 Conclusión de negocio

> La combinación **España + canal `web` + categoría Informática + mes de enero** es el motor real del negocio de RetailNova. El resto de combinaciones aporta de forma marginal.

---

## 🗺️ Parte 17 — Flujo completo del pipeline

```text
1. CSV anuales (ventas_2022, 2023, 2024)
        │
        ▼
2. DataFrame raw (lectura conjunta + inferSchema)
        │
        ▼
3. DataFrame normalizado (cast de tipos + trim/lower)
        │
        ▼
4. Dataset[VentaRaw] (tipado seguro Scala)
        │  map { ... clasificarVenta + necesitaRevision }
        ▼
5. Dataset[VentaEnriquecida] (importe bruto/desc/neto + segmento)
        │  toDF()
        ▼
6. DataFrame + UDF clasificarRiesgoUDF
        │  +columnas nativas (year/month/quarter/when)
        ▼
7. Capa analítica final (selección reducida)
        │  partitionBy(anio, mes)
        ▼
8. Parquet particionado (Data Lake)
        │  createOrReplaceTempView
        ▼
9. Spark SQL (consultas BI)
```

## 💡 Para recordar

> En un flujo real **no usamos Dataset porque sí**: lo usamos cuando necesitamos lógica de negocio tipada y segura con Scala. Después volvemos a DataFrame porque Spark SQL, las escrituras particionadas y las consultas analíticas trabajan mejor sobre DataFrames y tablas.